In [17]:
import pandas as pd
import numpy as np

np.random.seed(7)

today = pd.Timestamp("2026-09-28")
num_customers = 200
history_days = 180

rows = []
for i in range(1, num_customers + 1):
    cid = f"cust_{i:03d}"
    orders_per_month = np.random.randint(1, 6)
    n_orders = orders_per_month * 6
    days_ago = np.random.randint(0, history_days, size=n_orders)
    for d in days_ago:
        order_date = today - pd.Timedelta(days=int(d))
        order_value = round(np.random.uniform(10, 150), 2)
        rows.append([cid, order_date, order_value])

orders = pd.DataFrame(rows, columns=["customer_id", "order_date", "order_value"])

print(orders.shape)
orders.head()

(3528, 3)


,customer_id,order_date,order_value
0,cust_001,2026-09-03,34.37
1,cust_001,2026-07-23,120.71
2,cust_001,2026-04-30,141.27
3,cust_001,2026-06-17,72.84
4,cust_001,2026-06-28,63.85


In [18]:
orders["days_ago"] = (today - orders["order_date"]).dt.days

# Group A: 20 customers who stopped ordering (nothing in the last 45 days)
all_customers = orders["customer_id"].unique()
stopped = np.random.choice(all_customers, size=20, replace=False)
orders = orders[~(orders["customer_id"].isin(stopped) & (orders["days_ago"] < 45))]

# Group B: 15 customers who were ordering a lot but have slowed down
prior = orders[(orders["days_ago"] >= 30) & (orders["days_ago"] < 60)].groupby("customer_id").size()
candidates = [c for c in prior[prior >= 4].index if c not in stopped]
slowing = np.random.choice(candidates, size=min(15, len(candidates)), replace=False)

recent = orders["customer_id"].isin(slowing) & (orders["days_ago"] < 30)
drop = recent & (np.random.rand(len(orders)) < 0.75)
orders = orders[~drop].reset_index(drop=True)

print(len(stopped), len(slowing), orders.shape)

20 15 (3383, 4)


In [19]:
last_order = orders.groupby("customer_id")["days_ago"].min().rename("days_since_last_order")

recent_30 = orders[orders["days_ago"] < 30].groupby("customer_id").size().rename("orders_last_30")
prior_30 = orders[(orders["days_ago"] >= 30) & (orders["days_ago"] < 60)].groupby("customer_id").size().rename("orders_prev_30")

customers = pd.concat([last_order, recent_30, prior_30], axis=1).fillna(0)
customers["days_since_last_order"] = customers["days_since_last_order"].astype(int)

customers["signal_inactive"] = customers["days_since_last_order"] >= 30
customers["signal_drop"] = (customers["orders_prev_30"] >= 2) & (customers["orders_last_30"] <= 0.5 * customers["orders_prev_30"])

flagged = customers[customers["signal_inactive"] | customers["signal_drop"]]
planted = set(stopped) | set(slowing)

print("Flagged:", len(flagged))
print("Planted caught:", len(planted & set(flagged.index)), "of", len(planted))

Flagged: 90
Planted caught: 35 of 35


In [20]:
extras = flagged.loc[~flagged.index.isin(planted)]

print("Extras:", len(extras))
print(extras[["signal_inactive", "signal_drop"]].sum())
print(extras["days_since_last_order"].describe())

Extras: 55
signal_inactive    26
signal_drop        44
dtype: int64
count    55.000000
mean     28.581818
std      22.995725
min       0.000000
25%      10.000000
50%      27.000000
75%      41.000000
max      99.000000
Name: days_since_last_order, dtype: float64


In [21]:
span = orders.groupby("customer_id")["days_ago"].agg(["max", "min", "count"])
customers["avg_gap"] = (span["max"] - span["min"]) / (span["count"] - 1).replace(0, np.nan)
customers["overdue_ratio"] = customers["days_since_last_order"] / customers["avg_gap"]

customers["signal_inactive"] = (customers["days_since_last_order"] >= 30) & (customers["overdue_ratio"] >= 2)
customers["signal_drop"] = (customers["orders_prev_30"] >= 4) & (customers["orders_last_30"] <= 0.5 * customers["orders_prev_30"])

flagged = customers[customers["signal_inactive"] | customers["signal_drop"]]

print("Flagged:", len(flagged))
print("Stopped caught:", len(set(stopped) & set(flagged.index)), "of", len(stopped))
print("Slowing caught:", len(set(slowing) & set(flagged.index)), "of", len(slowing))

Flagged: 70
Stopped caught: 19 of 20
Slowing caught: 15 of 15


In [22]:
missed = set(stopped) - set(flagged.index)
print(customers.loc[list(missed)])

print("Customers with no orders at all:", num_customers - len(customers))

extras = flagged.loc[~flagged.index.isin(planted)]
print("Extras:", len(extras))
print(extras[["signal_inactive", "signal_drop"]].sum())

             days_since_last_order  orders_last_30  orders_prev_30  \
customer_id                                                          
cust_112                        74             0.0             0.0   

             signal_inactive  signal_drop  avg_gap  overdue_ratio  
customer_id                                                        
cust_112               False        False     44.0       1.681818  
Customers with no orders at all: 0
Extras: 36
signal_inactive    19
signal_drop        20
dtype: int64


In [23]:
customers["overdue_ratio"] = customers["overdue_ratio"].fillna(0)

inactive_part = (customers["overdue_ratio"] / 3).clip(upper=1)

drop_part = (1 - customers["orders_last_30"] / customers["orders_prev_30"].replace(0, np.nan)).clip(lower=0, upper=1)
drop_part = drop_part.where(customers["orders_prev_30"] >= 4, 0).fillna(0)

customers["risk_score"] = (50 * inactive_part + 50 * drop_part).round(1)

flagged = customers[customers["signal_inactive"] | customers["signal_drop"]]
ranked = flagged.sort_values("risk_score", ascending=False)

ranked_list = ranked[["days_since_last_order", "orders_prev_30", "orders_last_30",
                      "signal_inactive", "signal_drop", "risk_score"]].reset_index()
ranked_list.index = ranked_list.index + 1

top35 = set(ranked.head(35).index)
print("Planted customers in the top 35:", len(top35 & planted), "of 35")
ranked_list.head(15)

Planted customers in the top 35: 19 of 35


,customer_id,days_since_last_order,orders_prev_30,orders_last_30,signal_inactive,signal_drop,risk_score
1,cust_173,50,4.0,0.0,True,True,100.0
2,cust_122,43,6.0,0.0,True,True,100.0
3,cust_048,31,4.0,0.0,True,True,100.0
4,cust_049,45,4.0,0.0,True,True,100.0
5,cust_051,36,4.0,0.0,True,True,100.0
6,cust_082,32,6.0,0.0,True,True,100.0
7,cust_127,31,6.0,0.0,True,True,100.0
8,cust_010,28,6.0,1.0,False,True,91.7
9,cust_024,16,6.0,1.0,False,True,89.8
10,cust_075,31,4.0,0.0,True,True,86.4


In [24]:
customers["overdue_ratio"] = customers["overdue_ratio"].fillna(0)

inactive_part = (customers["overdue_ratio"] / 3).clip(upper=1)

drop_part = (1 - customers["orders_last_30"] / customers["orders_prev_30"].replace(0, np.nan)).clip(lower=0, upper=1)
drop_part = drop_part.where(customers["orders_prev_30"] >= 4, 0).fillna(0)

customers["risk_score"] = (50 * inactive_part + 50 * drop_part).round(1)

flagged = customers[customers["signal_inactive"] | customers["signal_drop"]]
ranked = flagged.sort_values("risk_score", ascending=False)

ranked_list = ranked[["days_since_last_order", "orders_prev_30", "orders_last_30",
                      "signal_inactive", "signal_drop", "risk_score"]].reset_index()
ranked_list.index = ranked_list.index + 1

top35 = set(ranked.head(35).index)
print("Planted customers in the top 35:", len(top35 & planted), "of 35")
ranked_list.head(15)

Planted customers in the top 35: 19 of 35


,customer_id,days_since_last_order,orders_prev_30,orders_last_30,signal_inactive,signal_drop,risk_score
1,cust_173,50,4.0,0.0,True,True,100.0
2,cust_122,43,6.0,0.0,True,True,100.0
3,cust_048,31,4.0,0.0,True,True,100.0
4,cust_049,45,4.0,0.0,True,True,100.0
5,cust_051,36,4.0,0.0,True,True,100.0
6,cust_082,32,6.0,0.0,True,True,100.0
7,cust_127,31,6.0,0.0,True,True,100.0
8,cust_010,28,6.0,1.0,False,True,91.7
9,cust_024,16,6.0,1.0,False,True,89.8
10,cust_075,31,4.0,0.0,True,True,86.4


In [25]:
ranked = flagged.sort_values(["risk_score", "days_since_last_order"], ascending=[False, False])

In [26]:
ranked_list[ranked_list["customer_id"] == "cust_112"]

,customer_id,days_since_last_order,orders_prev_30,orders_last_30,signal_inactive,signal_drop,risk_score


In [27]:
ranked_list = ranked[["days_since_last_order", "orders_prev_30", "orders_last_30",
                      "signal_inactive", "signal_drop", "risk_score"]].reset_index()
ranked_list.index = ranked_list.index + 1
ranked_list.head(15)

,customer_id,days_since_last_order,orders_prev_30,orders_last_30,signal_inactive,signal_drop,risk_score
1,cust_173,50,4.0,0.0,True,True,100.0
2,cust_049,45,4.0,0.0,True,True,100.0
3,cust_122,43,6.0,0.0,True,True,100.0
4,cust_051,36,4.0,0.0,True,True,100.0
5,cust_082,32,6.0,0.0,True,True,100.0
6,cust_048,31,4.0,0.0,True,True,100.0
7,cust_127,31,6.0,0.0,True,True,100.0
8,cust_010,28,6.0,1.0,False,True,91.7
9,cust_024,16,6.0,1.0,False,True,89.8
10,cust_075,31,4.0,0.0,True,True,86.4


In [28]:
ranked_list.to_csv("at_risk_customers.csv")
from google.colab import files
files.download("at_risk_customers.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>